In [1]:
import torch
import json
import pickle
import math
from pathlib import Path

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

with open(DATA_DIR / "vocab.json", encoding="utf-8") as f:
    word_to_id = json.load(f)

token_data = torch.load(
    DATA_DIR / "token_ids.pt",
    weights_only=True
)

train_ids = token_data["train"]
val_ids = token_data["validation"]
test_ids = token_data["test"]

# Only load pickle files that you created and trust.
with open(MODELS_DIR / "ngram_model.pkl", "rb") as f:
    ngram_model = pickle.load(f)

unigram_counts = ngram_model["unigram_counts"]
bigram_counts = ngram_model["bigram_counts"]
trigram_counts = ngram_model["trigram_counts"]

LAMBDA_1 = ngram_model["lambda1"]
LAMBDA_2 = ngram_model["lambda2"]
LAMBDA_3 = ngram_model["lambda3"]
ALPHA = ngram_model["alpha"]

VOCAB_SIZE = len(word_to_id)
PAD_ID = word_to_id["<pad>"]

print("Data and n-gram model loaded successfully!")
print("Vocabulary size:", VOCAB_SIZE)
print("Test tokens:", len(test_ids))
print("Interpolation weights:",
      LAMBDA_1, LAMBDA_2, LAMBDA_3)

Data and n-gram model loaded successfully!
Vocabulary size: 10000
Test tokens: 241211
Interpolation weights: 0.2 0.4 0.4


In [2]:
TOTAL_TOKENS = sum(unigram_counts.values())
PREDICTABLE_VOCAB_SIZE = VOCAB_SIZE - 1

def unigram_prob(w):
    if w == PAD_ID:
        return 0.0

    return (
        unigram_counts[w] + ALPHA
    ) / (
        TOTAL_TOKENS + ALPHA * PREDICTABLE_VOCAB_SIZE
    )


def interpolated_prob(w_prev2, w_prev1, w):
    if w == PAD_ID:
        return 0.0

    p1 = unigram_prob(w)

    bigram_context_count = unigram_counts[w_prev1]

    if bigram_context_count > 0:
        p2 = (
            bigram_counts[(w_prev1, w)]
            / bigram_context_count
        )
    else:
        p2 = 0.0

    trigram_context_count = bigram_counts[
        (w_prev2, w_prev1)
    ]

    if trigram_context_count > 0:
        p3 = (
            trigram_counts[(w_prev2, w_prev1, w)]
            / trigram_context_count
        )
    else:
        p3 = 0.0

    weight1 = LAMBDA_1
    weight2 = LAMBDA_2 if bigram_context_count > 0 else 0.0
    weight3 = LAMBDA_3 if trigram_context_count > 0 else 0.0

    total_weight = weight1 + weight2 + weight3

    return (
        weight1 * p1 +
        weight2 * p2 +
        weight3 * p3
    ) / total_weight

In [3]:
CONTEXT_LENGTH = 32

def calculate_chunk_aligned_perplexity(token_ids):
    total_log_prob = 0.0
    total_predictions = 0

    num_chunks = (len(token_ids) - 1) // CONTEXT_LENGTH

    for chunk_idx in range(num_chunks):
        start = chunk_idx * CONTEXT_LENGTH

        for offset in range(CONTEXT_LENGTH):
            target_pos = start + offset + 1
            target = token_ids[target_pos]

            if offset == 0:
                # Only one input token is available
                prev = token_ids[start]

                p1 = unigram_prob(target)

                context_count = unigram_counts[prev]

                if context_count > 0:
                    p2 = (
                        bigram_counts[(prev, target)]
                        / context_count
                    )

                    probability = (
                        LAMBDA_1 * p1 +
                        LAMBDA_2 * p2
                    ) / (LAMBDA_1 + LAMBDA_2)
                else:
                    probability = p1

            else:
                # Two preceding tokens available
                w1 = token_ids[target_pos - 2]
                w2 = token_ids[target_pos - 1]

                probability = interpolated_prob(
                    w1, w2, target
                )

            total_log_prob += math.log(probability)
            total_predictions += 1

    avg_loss = -total_log_prob / total_predictions
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity, total_predictions


aligned_loss, aligned_ppl, n_tokens = (
    calculate_chunk_aligned_perplexity(test_ids)
)

print(f"Aligned test loss: {aligned_loss:.4f}")
print(f"Aligned test perplexity: {aligned_ppl:.2f}")
print(f"Evaluated tokens: {n_tokens}")

Aligned test loss: 5.0650
Aligned test perplexity: 158.38
Evaluated tokens: 241184


In [4]:
import sys

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from language_models import (
    RNNLanguageModel,
    LSTMLanguageModel,
    TransformerLanguageModel
)

print("All model classes imported successfully!")

All model classes imported successfully!


In [5]:
import torch

DEVICE = torch.device("cpu")

# RNN
rnn_model = RNNLanguageModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=100,
    hidden_dim=128,
    num_layers=1,
    pad_id=PAD_ID
).to(DEVICE)

rnn_checkpoint = torch.load(
    MODELS_DIR / "rnn_best.pt",
    map_location=DEVICE,
    weights_only=True
)

rnn_model.load_state_dict(
    rnn_checkpoint["model_state_dict"]
)
rnn_model.eval()

# LSTM
lstm_model = LSTMLanguageModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=100,
    hidden_dim=128,
    num_layers=1,
    pad_id=PAD_ID
).to(DEVICE)

lstm_checkpoint = torch.load(
    MODELS_DIR / "lstm_best.pt",
    map_location=DEVICE,
    weights_only=True
)

lstm_model.load_state_dict(
    lstm_checkpoint["model_state_dict"]
)
lstm_model.eval()

# Transformer
transformer_model = TransformerLanguageModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=100,
    num_heads=4,
    num_layers=2,
    ff_dim=256,
    max_seq_length=32,
    dropout=0.1,
    pad_id=PAD_ID
).to(DEVICE)

transformer_checkpoint = torch.load(
    MODELS_DIR / "transformer_best.pt",
    map_location=DEVICE,
    weights_only=True
)

transformer_model.load_state_dict(
    transformer_checkpoint["model_state_dict"]
)
transformer_model.eval()

print("RNN loaded successfully!")
print("LSTM loaded successfully!")
print("Transformer loaded successfully!")

RNN loaded successfully!
LSTM loaded successfully!
Transformer loaded successfully!


In [6]:
models = {
    "RNN": rnn_model,
    "LSTM": lstm_model,
    "Transformer": transformer_model
}

for name, model in models.items():
    params = sum(p.numel() for p in model.parameters())
    size_mb = params * 4 / (1024 ** 2)

    print(
        f"{name:12s} | "
        f"Parameters: {params:,} | "
        f"FP32 weights: {size_mb:.2f} MiB"
    )

RNN          | Parameters: 2,319,440 | FP32 weights: 8.85 MiB
LSTM         | Parameters: 2,407,760 | FP32 weights: 9.18 MiB
Transformer  | Parameters: 2,197,912 | FP32 weights: 8.38 MiB


In [7]:
import torch
from torch.utils.data import DataLoader, TensorDataset

CONTEXT_LENGTH = 32
BATCH_SIZE = 32

# Use 313 chunks, each containing 32 predictions.
# This gives 10,016 target positions.
NUM_CHUNKS = 313

inputs = []
targets = []

for i in range(NUM_CHUNKS):
    start = i * CONTEXT_LENGTH

    x = test_ids[start:start + CONTEXT_LENGTH]
    y = test_ids[start + 1:start + CONTEXT_LENGTH + 1]

    inputs.append(x)
    targets.append(y)

inputs = torch.tensor(inputs, dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)

inference_dataset = TensorDataset(inputs, targets)

inference_loader = DataLoader(
    inference_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Input shape:", inputs.shape)
print("Target shape:", targets.shape)
print("Total target tokens:", targets.numel())
print("Inference batches:", len(inference_loader))

Input shape: torch.Size([313, 32])
Target shape: torch.Size([313, 32])
Total target tokens: 10016
Inference batches: 10


In [8]:
import time
import statistics

@torch.inference_mode()
def benchmark_inference(model, loader, repetitions=3):
    model.eval()

    # Warm-up run
    for x, y in loader:
        x = x.to(DEVICE)
        _ = model(x)

    times = []

    for _ in range(repetitions):
        start_time = time.perf_counter()

        for x, y in loader:
            x = x.to(DEVICE)

            logits = model(x)

        elapsed = time.perf_counter() - start_time
        times.append(elapsed)

    median_time = statistics.median(times)

    total_tokens = len(loader.dataset) * CONTEXT_LENGTH
    throughput = total_tokens / median_time

    return median_time, throughput

In [9]:
inference_results = {}

for name, model in models.items():

    elapsed, throughput = benchmark_inference(
        model,
        inference_loader,
        repetitions=3
    )

    inference_results[name] = {
        "inference_time_seconds": elapsed,
        "tokens_per_second": throughput
    }

    print(
        f"{name:12s} | "
        f"Time: {elapsed:.4f}s | "
        f"Throughput: {throughput:,.2f} tokens/s"
    )

RNN          | Time: 0.1980s | Throughput: 50,583.92 tokens/s
LSTM         | Time: 0.1414s | Throughput: 70,843.18 tokens/s
Transformer  | Time: 0.1201s | Throughput: 83,415.57 tokens/s


In [10]:
import time
import statistics

def benchmark_ngram(inputs, targets, repetitions=5):
    # Convert tensors to lists outside the timed section
    x_data = inputs.tolist()
    y_data = targets.tolist()

    def run_inference():
        total_probability = 0.0

        for x, y in zip(x_data, y_data):
            for position in range(len(x)):
                target = y[position]

                if position == 0:
                    # Only the first input token is available
                    prev = x[0]

                    p1 = unigram_prob(target)
                    context_count = unigram_counts[prev]

                    if context_count > 0:
                        p2 = (
                            bigram_counts[(prev, target)]
                            / context_count
                        )

                        probability = (
                            LAMBDA_1 * p1 +
                            LAMBDA_2 * p2
                        ) / (LAMBDA_1 + LAMBDA_2)
                    else:
                        probability = p1

                else:
                    probability = interpolated_prob(
                        x[position - 1],
                        x[position],
                        target
                    )

                total_probability += probability

        return total_probability

    # Warm-up
    run_inference()

    times = []

    for _ in range(repetitions):
        start = time.perf_counter()
        run_inference()
        times.append(time.perf_counter() - start)

    median_time = statistics.median(times)
    num_tokens = targets.numel()

    return median_time, num_tokens / median_time


ngram_time, ngram_throughput = benchmark_ngram(
    inputs,
    targets,
    repetitions=5
)

print(f"Trigram inference time: {ngram_time:.4f} seconds")
print(f"Trigram throughput: {ngram_throughput:,.2f} tokens/s")
print(f"Evaluated tokens: {targets.numel():,}")

Trigram inference time: 0.0170 seconds
Trigram throughput: 589,800.97 tokens/s
Evaluated tokens: 10,016


In [11]:
import json

comparison_results = {
    "Trigram": {
        "test_ppl": 158.38,
        "test_loss": 5.0650,
        "inference_time": ngram_time,
        "throughput": ngram_throughput,
        "memory_mb_approx": 320.13
    },
    "RNN": {
        "test_ppl": 128.39,
        "test_loss": 4.8550,
        "training_minutes": 23.56,
        "parameters": 2319440,
        **inference_results["RNN"]
    },
    "LSTM": {
        "test_ppl": 131.49,
        "test_loss": 4.8790,
        "training_minutes": 24.74,
        "parameters": 2407760,
        **inference_results["LSTM"]
    },
    "Transformer": {
        "test_ppl": 114.31,
        "test_loss": 4.7389,
        "training_minutes": 31.75,
        "parameters": 2197912,
        **inference_results["Transformer"]
    }
}

with open(
    RESULTS_DIR / "part1_comparison.json",
    "w"
) as f:
    json.dump(comparison_results, f, indent=2)

print("Part I comparison saved successfully!")

Part I comparison saved successfully!
